# Butterfly DDPM training

Clones the repo from GitHub, trains in Colab, and writes checkpoints to Google Drive.

Pick a new `RUN_NAME` for each experiment.

In [ ]:
import os
import shutil
from pathlib import Path

from google.colab import drive

drive.mount("/content/drive")

REPO_URL = "https://github.com/AAnanya19/Human-Faces-Generation-Diffusion-Models.git"
BRANCH = "main"
WORKDIR = Path("/content/Human-Faces-Generation-Diffusion-Models")

CHECKPOINTS_ROOT = Path("/content/drive/MyDrive/aml/ddpm_runs")
RUN_NAME = "butterfly_run_001"
RUN_DIR = CHECKPOINTS_ROOT / RUN_NAME

if WORKDIR.exists():
    shutil.rmtree(WORKDIR)

!git clone --branch "$BRANCH" "$REPO_URL" "$WORKDIR"

os.chdir(WORKDIR)
RUN_DIR.mkdir(parents=True, exist_ok=True)

print("Project root:", WORKDIR)
print("Run dir:", RUN_DIR)


##dependencies

In [ ]:
assert os.path.isfile("requirements.txt"), "Run the clone cell first."

!pip install -q datasets huggingface_hub tqdm matplotlib torchvision


## Training config

Set a fresh `RUN_NAME` for each experiment so checkpoints do not overwrite older runs.


In [ ]:
EPOCHS = 50
BATCH_SIZE = 32
IMAGE_SIZE = 64
LR = 1e-4
TIMESTEPS = 1000
WEIGHT_DECAY = 1e-4
GRAD_CLIP = 1.0
SAMPLE_EVERY = 10
NUM_SAMPLE_IMAGES = 8


## Run training


In [ ]:
import torch

device = "cuda" if torch.cuda.is_available() else "cpu"
print("torch:", torch.__version__)
print("device:", device)
if device != "cuda":
    raise RuntimeError("GPU is not active. In Colab, switch Runtime -> Change runtime type -> GPU.")

!python3 src/training/train.py \
    --epochs $EPOCHS \
    --batch_size $BATCH_SIZE \
    --image_size $IMAGE_SIZE \
    --lr $LR \
    --timesteps $TIMESTEPS \
    --weight_decay $WEIGHT_DECAY \
    --grad_clip $GRAD_CLIP \
    --sample_every $SAMPLE_EVERY \
    --num_sample_images $NUM_SAMPLE_IMAGES \
    --device cuda \
    --save_dir "$RUN_DIR"


## Inspect saved checkpoints


In [ ]:
print("Checkpoint files:", sorted(str(p.name) for p in RUN_DIR.glob("*.pth")))
print("Sample grids:", sorted(str(p.name) for p in (RUN_DIR / "generated_samples").glob("*.png")))
print("Metadata file exists:", (RUN_DIR / "run_config.json").is_file())
print((RUN_DIR / "run_config.json").read_text())


The visualization notebook should point to the same `RUN_DIR`, for example `RUN_DIR / "ddpm_final.pth"`. During training, preview grids are also saved in `RUN_DIR / "generated_samples"`.
